In [1]:
import os
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sqlalchemy import create_engine

# 1. Connect to SQLite
db_path = os.path.join("..", "data", "bird_monitoring.db")
engine = create_engine(f"sqlite:///{db_path}")
df = pd.read_sql("SELECT * FROM bird_observations", con=engine)
df['Date'] = pd.to_datetime(df['Date'])

# 2. Extract Observation Hour
df['Observation_Hour'] = df['Start_Time'].astype(str).str.extract(r'(\d{1,2}):')[0].astype(float)

# 3. Habitat Comparison Figure
hab = df.groupby('Location_Type').agg(
    Total_Sightings=('Common_Name', 'count'),
    Unique_Species=('Scientific_Name', 'nunique')
).reset_index()

fig1 = make_subplots(rows=1, cols=2, subplot_titles=("Total Sightings", "Species Richness"))
fig1.add_trace(go.Bar(x=hab['Location_Type'], y=hab['Total_Sightings'], text=hab['Total_Sightings'], textposition='auto', marker_color='#2ca02c'), row=1, col=1)
fig1.add_trace(go.Bar(x=hab['Location_Type'], y=hab['Unique_Species'], text=hab['Unique_Species'], textposition='auto', marker_color='#1f77b4'), row=1, col=2)
fig1.update_layout(title_text="Habitat Comparison: Forest vs Grassland", showlegend=False, height=400)
fig1.show()

# 4. Conservation Watchlist Figure
watch = df[df['PIF_Watchlist_Status'].isin([True, 1, 'True', 'TRUE'])]['Common_Name'].value_counts().head(10).reset_index()
watch.columns = ['Species', 'Sightings']
fig2 = px.bar(watch, x='Sightings', y='Species', orientation='h', color='Sightings',
              title="Top 10 PIF Watchlist Species at Risk", color_continuous_scale='Reds')
fig2.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig2.show()

# 5. Diurnal Activity Patterns Figure
fig3 = px.histogram(df.dropna(subset=['Observation_Hour']), x='Observation_Hour', color='Location_Type',
                    barmode='group', nbins=14, title="Diurnal Observation Patterns")
fig3.update_layout(xaxis_title="Hour of Day (24-hr)", yaxis_title="Sightings Count", height=400)
fig3.show()